In [0]:
from pyspark.ml.feature import VectorAssembler
v = VectorAssembler(inputCols=["a"], outputCol="features")
print("Works!")

In [0]:

import os
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/workspace/default/uploads/sparkml_cache'

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, hour
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import mlflow

from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

np.random.seed(14)

In [0]:
# Load Silver Delta Table and prepare data
raw_df = spark.table("workspace.default.silver_accidents_hpms")
print(f"Full silver table: {raw_df.count()} rows")

# Limiting to 50k rows to stay under serverless 1GB cache limit
df_mllib = raw_df.limit(50000) \
    .select(
        col("Distance_mi").cast("double"),
        col("Temperature_F").cast("double"),
        col("Visibility_mi").cast("double"),
        col("Wind_Speed_mph").cast("double"),
        col("Start_Lat").cast("double"),
        col("Start_Lng").cast("double"),
        col("hour").cast("integer"),
        col("day_of_week").cast("integer"),
        col("month").cast("integer"),
        col("AADT").cast("double"),
        col("SPEED_LIMIT").cast("double"),
        col("THROUGH_LANES").cast("double"),
        col("F_SYSTEM").cast("double"),
        col("ACCESS_CONTROL").cast("double"),
        when(col("Severity") <= 2, 0).otherwise(1).alias("label")
    ).dropna()

print(f"Data ready for training: {df_mllib.count()} rows")

In [0]:
# Clearing cached models from previous runs
try:
    del cv_model
    del best_model
    del best_rf
    del predictions
    del model
except NameError:
    pass

import gc
gc.collect()
print("Cache cleared")

In [0]:
# Splitting before training to avoid data leakage
train_df, test_df = df_mllib.randomSplit([0.8, 0.2], seed=14)
print(f"Train: {train_df.count()}  Test: {test_df.count()}")

In [0]:
feature_cols = [
    "Distance_mi", "Temperature_F", "Visibility_mi", "Wind_Speed_mph",
    "Start_Lat", "Start_Lng", "hour", "day_of_week", "month",
    "AADT", "SPEED_LIMIT", "THROUGH_LANES", "F_SYSTEM", "ACCESS_CONTROL"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled")

rf = RandomForestClassifier(
    featuresCol="scaled",
    labelCol="label",
    seed=14
)

pipeline = Pipeline(stages=[assembler, scaler, rf])

# CrossValidator for hyperparameter tuning
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [50]) \
    .addGrid(rf.maxDepth, [10]) \
    .build()

evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

crossValidator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    seed=14,
    parallelism=1
)

In [0]:
with mlflow.start_run(run_name="Phase3_RandomForest_HPMS"):
    print("Training Random Forest with CrossValidator...")
    cv_model = crossValidator.fit(train_df)

    # Get best model details
    best_rf = cv_model.bestModel.stages[-1]
    print(f"Best numTrees: {best_rf.getNumTrees}")
    print(f"Best maxDepth: {best_rf.getMaxDepth()}")

    # Log to MLflow
    mlflow.log_param("numTrees", best_rf.getNumTrees)
    mlflow.log_param("maxDepth", best_rf.getMaxDepth())
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("data_source", "accidents + HPMS")

    print("\nRandom Forest training complete!")

In [0]:
# Evaluate the best model
best_model = cv_model.bestModel
best_rf = best_model.stages[-1]

predictions = best_model.transform(test_df)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

metrics_data = {
    "Metric": ["F1 Score", "Accuracy", "Weighted Precision", "Weighted Recall"],
    "Value": [
        round(evaluator.evaluate(predictions, {evaluator.metricName: "f1"}), 4),
        round(evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"}), 4),
        round(evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"}), 4),
        round(evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"}), 4)
    ]
}

print("="*50)
print("             PHASE 3 RANDOM FOREST EVALUATION       ")
print("="*50)
print(f"Model:           Random Forest Classifier")
print(f"Best numTrees:   {best_rf.getNumTrees}")
print(f"Best maxDepth:   {best_rf.getMaxDepth()}")
print("-" * 50)

results_df = pd.DataFrame(metrics_data)
print(results_df.to_string(index=False))
print("-" * 50)

# Phase 2 vs Phase 3 comparison
phase2_f1 = 0.59  # update with your actual Phase 2 RF F1
improvement = metrics_data["Value"][0] - phase2_f1

print(f"Phase 2 F1 Score:    {phase2_f1}")
print(f"Phase 3 F1 Score:    {metrics_data['Value'][0]}")
print(f"Difference:          {round(improvement, 4)}")
print("="*50)

In [0]:
# sanity check
print(f"Predictions count: {predictions.count()}")
print(f"Test df count: {test_df.count()}")


In [0]:
# Confusion Matrix
preds_pd = predictions.select("label", "prediction").toPandas()
y_true = preds_pd["label"].values
y_pred = preds_pd["prediction"].values

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Minor", "Severe"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Random Forest - Confusion Matrix (Phase 3 + HPMS)")
plt.tight_layout()
plt.savefig("/tmp/rf_cm.png", dpi=150)
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Minor", "Severe"]))

In [0]:
# Feature Importance
fi = pd.Series(
    best_rf.featureImportances.toArray(),
    index=feature_cols
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
fi.sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_title("Feature Importances - RF with HPMS Features")
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.savefig("/tmp/rf_feature_importance.png", dpi=150)
plt.show()

print("\nTop features:")
print(fi)

In [0]:
# Save model
model_path = "/Volumes/workspace/default/uploads/models/phase3_random_forest"
best_model.write().overwrite().save(model_path)
print(f"Model saved to: {model_path}")